# TM10007 Assignment template

In [63]:
# Run this to use from colab environment
!pip install -q --upgrade git+https://github.com/jveenland/tm10007_ml.git

## Importing packages 


In [72]:
# General packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets as ds
import seaborn as sns

# Classifiers
from sklearn import model_selection
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm

# statistics
from scipy.stats import zscore, shapiro

## Data loading and cleaning

Below are functions to load the dataset of your choice. After that, it is all up to you to create and evaluate a classification method. Beware, there may be missing values in these datasets. Good luck!

In [65]:
from worcliver.load_data import load_data

# Load the data
data = load_data()

# Replace the labels with binary labels
data_binary = data.copy()
data_binary["label"] = data_binary["label"].map({"malignant": 1, "benign": 0})

# Print number of samples and columns
print(f"The number of samples: {len(data_binary.index)}")
print(f"The number of columns: {len(data_binary.columns)}")
print(f"The number of malignant samples: {sum(data_binary['label']==1)}")
print(f"The number of benign samples: {sum(data_binary['label']==0)}")

The number of samples: 186
The number of columns: 494
The number of malignant samples: 94
The number of benign samples: 92


## Data splitting

In [66]:
# Split the dataset in features and labels
X = data_binary.drop(columns=["label"])
y = data_binary["label"]

# Split the dataset in train and test part
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Print the number of samples in the train and test set
print(f"The number of samples in the train set: {len(X_train.index)}")
print(f"The number of samples in the test set: {len(X_test.index)}")

The number of samples in the train set: 148
The number of samples in the test set: 38


## Pre-Processing


In [70]:
# Checking for duplicate samples
duplicate_samples = data_binary[data_binary.duplicated()]
if len(duplicate_samples) > 0:
    print(f"Number of duplicate rows: {len(duplicate_samples)}")
else:
    print("No duplicate sample found.")   
# Remove duplicate samples
data_binary = data_binary.drop_duplicates()

# Checking for duplicate features
duplicate_features = data_binary.columns[data_binary.columns.duplicated()]
if len(duplicate_features) > 0:
    print(f"Number of duplicate features: {len(duplicate_features)}")
    print("Duplicate feature names:")
    print(list(duplicate_features))
    # Remove duplicate features
    data_binary = data_binary.loc[:, ~data_binary.columns.duplicated()]   
else:
    print("No duplicate features found.")

# Checking for constant features
constant_features = data_binary.columns[data_binary.nunique() == 1]
print(f"Number of constant features: {len(constant_features)}")
if len(constant_features) > 0:
    print(f"Constant features names: {list(constant_features)}")
    data_binary.drop(columns=constant_features, inplace=True)

# Checking for missing data
# Check for missing values
print(f"Amount of missing data: {data_binary.isna().sum().sum()}")
# Check for infinite values
print(f"Amount of infinite data: {data_binary.map(np.isinf).sum().sum()}")


No duplicate sample found.
No duplicate features found.
Number of constant features: 0
Amount of missing data: 0
Amount of infinite data: 0


## Scaling 
##### First check the distribution of the data, we check for normal and not normal distribution and we check the data on outliers

In [83]:
k = 1.5  # IQR factor
n_outliers = {}

X_train_out = X_train.copy()
X_test_out = X_test.copy()

for feature in X_train_out.columns:
    q25, q75 = np.percentile(X_train_out[feature], 25), np.percentile(X_train_out[feature], 75)
    iqr = q75 - q25
    cut_off = iqr * k
    lower, upper = q25 - cut_off, q75 + cut_off

    # Count number of outliers
    outliers = [x for x in X_train_out[feature] if x < lower or x > upper]
    n_outliers[feature] = len(outliers)

    # Clipping
    X_train_out[feature] = np.clip(X_train_out[feature], lower, upper)
    X_test_out[feature] = np.clip(X_test_out[feature], lower, upper)

# Result 
print("Aantal outliers per feature:", n_outliers)
print("Totaal aantal outliers:", sum(n_outliers.values()))
print("Max outliers in één feature:", max(n_outliers.values()))


Aantal outliers per feature: {'PREDICT_original_sf_compactness_avg_2.5D': 7, 'PREDICT_original_sf_compactness_std_2.5D': 16, 'PREDICT_original_sf_rad_dist_avg_2.5D': 5, 'PREDICT_original_sf_rad_dist_std_2.5D': 6, 'PREDICT_original_sf_roughness_avg_2.5D': 4, 'PREDICT_original_sf_roughness_std_2.5D': 10, 'PREDICT_original_sf_convexity_avg_2.5D': 10, 'PREDICT_original_sf_convexity_std_2.5D': 17, 'PREDICT_original_sf_cvar_avg_2.5D': 5, 'PREDICT_original_sf_cvar_std_2.5D': 12, 'PREDICT_original_sf_prax_avg_2.5D': 1, 'PREDICT_original_sf_prax_std_2.5D': 1, 'PREDICT_original_sf_evar_avg_2.5D': 2, 'PREDICT_original_sf_evar_std_2.5D': 10, 'PREDICT_original_sf_solidity_avg_2.5D': 8, 'PREDICT_original_sf_solidity_std_2.5D': 16, 'PREDICT_original_sf_area_avg_2.5D': 16, 'PREDICT_original_sf_area_max_2.5D': 12, 'PREDICT_original_sf_area_min_2.5D': 12, 'PREDICT_original_sf_area_std_2.5D': 12, 'PREDICT_original_sf_volume_2.5D': 17, 'PREDICT_original_of_theta_x': 0, 'PREDICT_original_of_theta_y': 8, 'P